# Ensemble Framework for PTM Prediction

Combines predictions from multiple models:
- Random Forest
- XGBoost
- CNN (from teammate, optional)
- GRU (from teammate, optional)

## Ensemble Strategies:
1. Simple averaging
2. Weighted averaging (equal weights)
3. Optimized weights (grid search)

## Configuration

In [4]:
# File paths
VAL_FILE = "../output/val_with_features_split.csv"
RF_PREDICTIONS = "../output/rf_val_predictions.csv"
XGB_PREDICTIONS = "../output/xgb_val_predictions.csv"
CNN_PREDICTIONS = "../output/cnn_val_predictions.csv"  # Optional
GRU_PREDICTIONS = "../output/gru_val_predictions.csv"  # Optional

OUTPUT_WEIGHTS = "../output/ensemble_weights.pkl"
OUTPUT_PREDICTIONS = "../output/ensemble_val_predictions.csv"

## Import Libraries

In [5]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import f1_score, roc_auc_score, hamming_loss
import os
import warnings
warnings.filterwarnings("ignore")

## Load Predictions

In [6]:
val_df = pd.read_csv(VAL_FILE)
label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]
y_true = val_df[label_cols].values

predictions = {}

# Load RF predictions
if os.path.exists(RF_PREDICTIONS):
    rf = pd.read_csv(RF_PREDICTIONS)
    predictions["rf"] = rf[["rf_glut_proba", "rf_nitro_proba", "rf_palm_proba"]].values
    print("✓ Loaded RF predictions")

# Load XGBoost predictions
if os.path.exists(XGB_PREDICTIONS):
    xgb = pd.read_csv(XGB_PREDICTIONS)
    predictions["xgb"] = xgb[["xgb_glut_proba", "xgb_nitro_proba", "xgb_palm_proba"]].values
    print("✓ Loaded XGBoost predictions")

# Load CNN predictions (optional)
if os.path.exists(CNN_PREDICTIONS):
    cnn = pd.read_csv(CNN_PREDICTIONS)
    predictions["cnn"] = cnn[["cnn_glut_proba", "cnn_nitro_proba", "cnn_palm_proba"]].values
    print("✓ Loaded CNN predictions")

# Load GRU predictions (optional)
if os.path.exists(GRU_PREDICTIONS):
    gru = pd.read_csv(GRU_PREDICTIONS)
    predictions["gru"] = gru[["gru_glut_proba", "gru_nitro_proba", "gru_palm_proba"]].values
    print("✓ Loaded GRU predictions")

print(f"\nTotal models: {len(predictions)}")

FileNotFoundError: [Errno 2] No such file or directory: '../output/val_with_features_split.csv'

## Ensemble Evaluation Function

In [ ]:
def evaluate_ensemble(y_true, y_pred_proba, threshold=0.5):
    """Evaluate ensemble predictions."""
    y_pred = (y_pred_proba > threshold).astype(int)
    
    print(f"\n{"Label":<30} {"F1":<12} {"AUC":<12}")
    print("-" * 54)
    
    results = {}
    for idx, label in enumerate(label_cols):
        f1 = f1_score(y_true[:, idx], y_pred[:, idx])
        auc = roc_auc_score(y_true[:, idx], y_pred_proba[:, idx])
        results[label] = {"f1": f1, "auc": auc}
        print(f"{label:<30} {f1:<12.4f} {auc:<12.4f}")
    
    macro_f1 = np.mean([r["f1"] for r in results.values()])
    macro_auc = np.mean([r["auc"] for r in results.values()])
    
    print(f"\n{"Macro Average":<30} {macro_f1:<12.4f} {macro_auc:<12.4f}")
    
    return macro_f1, macro_auc

## Strategy 1: Simple Average

In [ ]:
print("="*80)
print("STRATEGY 1: SIMPLE AVERAGE")
print("="*80)

pred_avg = np.mean(list(predictions.values()), axis=0)
f1_avg, auc_avg = evaluate_ensemble(y_true, pred_avg)

## Strategy 2: Find Optimal Weights

In [ ]:
print("\n" + "="*80)
print("STRATEGY 2: OPTIMIZED WEIGHTS (Grid Search)")
print("="*80)

model_names = list(predictions.keys())
n_models = len(model_names)

best_f1 = 0
best_weights = None

if n_models == 2:
    # 2 models: try all combinations
    for w1 in [i/10 for i in range(0, 11)]:
        w2 = 1.0 - w1
        weights = {model_names[0]: w1, model_names[1]: w2}
        
        weighted_sum = sum(weights[m] * predictions[m] for m in model_names)
        f1, _ = evaluate_ensemble(y_true, weighted_sum)
        
        if f1 > best_f1:
            best_f1 = f1
            best_weights = weights
            
elif n_models == 3:
    # 3 models: sample combinations
    for w1 in [i/10 for i in range(0, 11, 2)]:
        for w2 in [i/10 for i in range(0, 11, 2)]:
            w3 = 1.0 - w1 - w2
            if w3 < 0:
                continue
            
            weights = {model_names[0]: w1, model_names[1]: w2, model_names[2]: w3}
            weighted_sum = sum(weights[m] * predictions[m] for m in model_names)
            f1, _ = evaluate_ensemble(y_true, weighted_sum)
            
            if f1 > best_f1:
                best_f1 = f1
                best_weights = weights

print(f"\nBest weights (F1={best_f1:.4f}):")
for model, weight in best_weights.items():
    print(f"  {model}: {weight:.2f}")

## Final Ensemble with Optimal Weights

In [ ]:
print("\n" + "="*80)
print("FINAL ENSEMBLE EVALUATION")
print("="*80)

pred_opt = sum(best_weights[m] * predictions[m] for m in model_names)
f1_opt, auc_opt = evaluate_ensemble(y_true, pred_opt)

## Save Ensemble

In [ ]:
# Save weights
with open(OUTPUT_WEIGHTS, "wb") as f:
    pickle.dump(best_weights, f)
print(f"✓ Saved {OUTPUT_WEIGHTS}")

# Save predictions
pred_df = pd.DataFrame({
    "ID": val_df["ID"],
    "ensemble_glut_proba": pred_opt[:, 0],
    "ensemble_nitro_proba": pred_opt[:, 1],
    "ensemble_palm_proba": pred_opt[:, 2]
})
pred_df.to_csv(OUTPUT_PREDICTIONS, index=False)
print(f"✓ Saved {OUTPUT_PREDICTIONS}")

print(f"\nFinal Performance: F1={f1_opt:.4f}, AUC={auc_opt:.4f}")